In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

df = pd.read_csv(r"C:\Users\joshh\Machine_Learning\Audiology_Project\Audio_Data.csv")
df['INVESTDATE'] = pd.to_datetime(df['INVESTDATE'],format="%d/%m/%Y")
LeftSC = ['L250C','L500C','L750C','L1000C','L1500C','L2000C','L3000C','L4000C','L6000C','L8000C']
RightSC = ['R250C','R500C','R750C','R1000C','R1500C','R2000C','R3000C','R4000C','R6000C',"R8000C"]
df[LeftSC] = df[LeftSC].replace(0, np.nan)
df[RightSC] = df[RightSC].replace(0, np.nan)
df['Right_Mean'] = df[RightSC].mean(axis=1)
df['Left_Mean'] = df[LeftSC].mean(axis=1)
df = df.sort_values(
    by=['Unique PID', 'INVESTDATE', 'CURVE'], 
    ascending=[True, True, True]
)
df['_nan_count'] = df.isnull().sum(axis=1)
df = df.loc[df.groupby(['Unique PID', 'INVESTDATE', 'CURVE'])['_nan_count'].idxmin()]
df = df.drop(columns=['_nan_count'])

df['next_PID'] = df['Unique PID'].shift(-1)
df['next_DATE'] = df['INVESTDATE'].shift(-1)
df['next_CURVE'] = df['CURVE'].shift(-1)
df['next_Right_Mean'] = df['Right_Mean'].shift(-1)
df['next_Left_Mean'] = df['Left_Mean'].shift(-1)
df['BR250C'] = df['R250C'].shift(-1)
df['BR500C'] = df['R500C'].shift(-1)
df['BR750C'] = df['R750C'].shift(-1)
df['BR1000C'] = df['R1000C'].shift(-1)
df['BR1500C'] = df['R1500C'].shift(-1)
df['BR2000C'] = df['R2000C'].shift(-1)
df['BR3000C'] = df['R3000C'].shift(-1)
df['BR4000C'] = df['R4000C'].shift(-1)
df['BR6000C'] = df['R6000C'].shift(-1)
df['BR8000C'] = df['R8000C'].shift(-1)
df['BL250C'] = df['L250C'].shift(-1)
df['BL500C'] = df['L500C'].shift(-1)
df['BL750C'] = df['L750C'].shift(-1)
df['BL1000C'] = df['L1000C'].shift(-1)
df['BL1500C'] = df['L1500C'].shift(-1)
df['BL2000C'] = df['L2000C'].shift(-1)
df['BL3000C'] = df['L3000C'].shift(-1)
df['BL4000C'] = df['L4000C'].shift(-1)
df['BL6000C'] = df['L6000C'].shift(-1)
df['BL8000C'] = df['L8000C'].shift(-1)

BLeftSC = ['BL250C','BL500C','BL750C','BL1000C','BL1500C','BL2000C','BL3000C','BL4000C','BL6000C','BL8000C']
BRightSC = ['BR250C','BR500C','BR750C','BR1000C','BR1500C','BR2000C','BR3000C','BR4000C','BR6000C',"BR8000C"]

condition = (
    (df['Unique PID'] == df['next_PID']) &
    (df['INVESTDATE'] == df['next_DATE']) &
    (df['CURVE'] == 0) &
    (df['next_CURVE'] == 10)
)

df['RAC_BC'] = pd.NA
df['LAC_BC'] = pd.NA
df.loc[condition, 'RAC_BC'] = np.abs(df['Right_Mean'] - df['next_Right_Mean'])
df.loc[condition, 'LAC_BC'] = np.abs(df['Left_Mean'] - df['next_Left_Mean'])
df = df.drop(columns=['next_PID', 'next_DATE', 'next_CURVE', 'next_Right_Mean', 'next_Left_Mean'])
dfdropped = df.dropna(subset=['RAC_BC', 'LAC_BC'], how='all').reset_index(drop=True)
dfdropped['Ear_Asymmetry'] = np.abs(dfdropped['Right_Mean'] - dfdropped['Left_Mean'])
df_end = dfdropped.copy()
df_end['RAC_BC'] = pd.to_numeric(df_end['RAC_BC'], errors='coerce')
df_end['LAC_BC'] = pd.to_numeric(df_end['LAC_BC'], errors='coerce')
df_end

LeftSC = [
    "L250C",
    "L500C",
    "L750C",
    "L1000C",
    "L1500C",
    "L2000C",
    "L3000C",
    "L4000C",
    "L6000C",
    "L8000C",
]
RightSC = [
    "R250C",
    "R500C",
    "R750C",
    "R1000C",
    "R1500C",
    "R2000C",
    "R3000C",
    "R4000C",
    "R6000C",
    "R8000C",
]

BLeftSC = [
    "BL250C",
    "BL500C",
    "BL750C",
    "BL1000C",
    "BL1500C",
    "BL2000C",
    "BL3000C",
    "BL4000C",
    "BL6000C",
    "BL8000C",
]
BRightSC = [
    "BR250C",
    "BR500C",
    "BR750C",
    "BR1000C",
    "BR1500C",
    "BR2000C",
    "BR3000C",
    "BR4000C",
    "BR6000C",
    "BR8000C",
]


LLeftSC = [
    "L250C",
    "L500C",
    "L750C",
    "L1000C",
    "L1500C"
]
LRightSC = [
    "R250C",
    "R500C",
    "R750C",
    "R1000C",
    "R1500C"
]
HLeftSC = [
    "L2000C",
    "L3000C",
    "L4000C",
    "L6000C",
    "L8000C",
]
HRightSC = [
    "R2000C",
    "R3000C",
    "R4000C",
    "R6000C",
    "R8000C",
]
HBLeftSC = [
    "BL2000C",
    "BL3000C",
    "BL4000C",
    "BL6000C",
    "BL8000C",
]
HBRightSC = [
    "BR2000C",
    "BR3000C",
    "BR4000C",
    "BR6000C",
    "BR8000C",
]
LBLeftSC = [
    "BL250C",
    "BL500C",
    "BL750C",
    "BL1000C",
    "BL1500C"
]
LBRightSC = [
    "BR250C",
    "BR500C",
    "BR750C",
    "BR1000C",
    "BR1500C"
]
df_end['BRight_Mean'] = df_end[BRightSC].mean(axis=1)
df_end['BLeft_Mean'] = df_end[BLeftSC].mean(axis=1)
df_end['HBRight_Mean'] = df_end[HBRightSC].mean(axis=1)
df_end['HBLeft_Mean'] = df_end[HBLeftSC].mean(axis=1)
df_end['HRight_Mean'] = df_end[HRightSC].mean(axis=1)
df_end['HLeft_Mean'] = df_end[HLeftSC].mean(axis=1)
df_end['LBRight_Mean'] = df_end[LBRightSC].mean(axis=1)
df_end['LBLeft_Mean'] = df_end[LBLeftSC].mean(axis=1)
df_end['LRight_Mean'] = df_end[LRightSC].mean(axis=1)
df_end['LLeft_Mean'] = df_end[LLeftSC].mean(axis=1)

df_test = df_end.copy().reset_index(drop=True)
df_test = df_test[ ~( (df_test['Treatment Cat'] == 'No Treatment') & ((df_test['Right_Mean'] > 30) | (df_test['Left_Mean'] > 30)) ) ]


frequency_cols = [col for col in df_test.columns if (col.startswith('R') or col.startswith('L') or col.startswith('B')) 
                  and col not in ['Right_Mean', 'Left_Mean', 'RAC_BC', 'LAC_BC','Ear_Asymmetry',
                                  'BLeft_Mean','BRight_Mean',"HBRight_Mean",	"HBLeft_Mean" ,	"HRight_Mean" ,	"HLeft_Mean" ,	"LBRight_Mean", "LBLeft_Mean" ,	"LRight_Mean",	"LLeft_Mean"]]

engineered_cols = ['Right_Mean', 'Left_Mean', 'RAC_BC', 'LAC_BC','Ear_Asymmetry',
                                  'BLeft_Mean','BRight_Mean',"HBRight_Mean",	"HBLeft_Mean" ,	"HRight_Mean" ,	"HLeft_Mean" ,	"LBRight_Mean", 	"LBLeft_Mean" ,	"LRight_Mean",	"LLeft_Mean"]
feature_cols = frequency_cols + engineered_cols



X = df_test[feature_cols]
y = df_test['Treatment Cat']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
target_names = label_encoder.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=1415, stratify=y_encoded
)

model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=15,
    learning_rate=0.1,
    random_state=1415,
    eval_metric='mlogloss'
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}\n")

print(classification_report(y_test, y_pred, target_names=target_names))

Accuracy: 0.91

              precision    recall  f1-score   support

     BCI_MEI       0.80      0.57      0.66      1578
          CI       0.82      0.70      0.76      1245
          HA       0.89      0.88      0.88      9044
No Treatment       0.94      1.00      0.97     13694

    accuracy                           0.91     25561
   macro avg       0.86      0.79      0.82     25561
weighted avg       0.91      0.91      0.91     25561

